# 🏥 Chest X-Ray Pneumonia Classifier
## Medical Training Pipeline — Phases 1 to 4

**Dataset:** Chest X-Ray Images (Pneumonia) — Kaggle  
**Task:** Binary Classification — NORMAL vs PNEUMONIA  
**Backbone:** DenseNet121 (ImageNet pretrained)  
**Explainability:** Grad-CAM  

> ⚠️ Before running: update the 3 dataset paths in **Cell 2** to match your machine.

In [ ]:
# ============================================================
#  CELL 1 — INSTALL & IMPORTS
# ============================================================
import subprocess, sys

for pkg in [
    "torch", "torchvision", "opencv-python-headless",
    "numpy", "matplotlib", "Pillow", "scikit-learn",
    "tqdm", "seaborn"
]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q"])

import os, cv2, random, warnings, time, copy, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import models, transforms
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler)
from sklearn.metrics import (
    confusion_matrix, f1_score,
    roc_auc_score, roc_curve)

warnings.filterwarnings("ignore")
print("✅ All packages installed and imported.")

In [ ]:
# ============================================================
#  CELL 2 — GLOBAL CONFIG
#  ⚠️  Change the 3 paths below to match your machine
# ============================================================

TRAIN_DIR = Path(r"C:\Users\Admin\Downloads\chest_xray\train")
VAL_DIR   = Path(r"C:\Users\Admin\Downloads\chest_xray\val")
TEST_DIR  = Path(r"C:\Users\Admin\Downloads\chest_xray\test")

IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_WORKERS   = 0        # Windows-safe (set to 4 on Linux/Mac)
NUM_EPOCHS    = 15
LEARNING_RATE = 1e-4
NUM_CLASSES   = 2
SEED          = 42
PATIENCE      = 5        # early stopping patience

CLASS_NAMES  = ["NORMAL", "PNEUMONIA"]
CLASS_TO_IDX = {"NORMAL": 0, "PNEUMONIA": 1}
IDX_TO_CLASS = {0: "NORMAL", 1: "PNEUMONIA"}
COLORS       = {"NORMAL": "#2ECC71", "PNEUMONIA": "#E74C3C"}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ Config ready")
print(f"   Device    : {DEVICE}")
print(f"   Train dir : {TRAIN_DIR}")
print(f"   Val dir   : {VAL_DIR}")
print(f"   Test dir  : {TEST_DIR}")

print("\n📁 Folder check:")
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if d.exists():
        classes = [x.name for x in d.iterdir() if x.is_dir()]
        counts  = {c: len(list((d/c).glob("*.*")))
                   for c in classes}
        print(f"   ✅ {d.name:<8} → {counts}")
    else:
        print(f"   ❌ NOT FOUND: {d}")

In [ ]:
# ============================================================
#  CELL 3 — LAPLACIAN QA FILTER & PATH COLLECTION
# ============================================================

QA_BLUR_THRESHOLD = 20.0   # images below this score are rejected

def laplacian_variance(img_path: str) -> float:
    """
    Sharpness score using Laplacian variance.
    High value = sharp image (good).
    Low value  = blurry image (reject).
    """
    gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        return 0.0
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def qa_filter(paths: list, labels: list,
              threshold: float = QA_BLUR_THRESHOLD):
    """Remove blurry or unreadable images from the dataset."""
    clean_paths, clean_labels, rejected = [], [], 0
    for path, label in zip(paths, labels):
        if laplacian_variance(path) >= threshold:
            clean_paths.append(path)
            clean_labels.append(label)
        else:
            rejected += 1
    return clean_paths, clean_labels, rejected


def collect_paths(split_dir: Path):
    """Walk NORMAL/ and PNEUMONIA/ → flat path + label lists."""
    paths, labels = [], []
    for class_name, label in CLASS_TO_IDX.items():
        class_dir = split_dir / class_name
        if not class_dir.exists():
            print(f"  ⚠️  Missing: {class_dir}")
            continue
        for p in sorted(class_dir.iterdir()):
            if p.suffix.lower() in {".jpeg", ".jpg", ".png"}:
                paths.append(str(p))
                labels.append(label)
    return paths, labels


print("📁 Collecting image paths ...")
raw_train_paths, raw_train_labels = collect_paths(TRAIN_DIR)
raw_val_paths,   raw_val_labels   = collect_paths(VAL_DIR)
raw_test_paths,  raw_test_labels  = collect_paths(TEST_DIR)

print(f"   Raw → Train:{len(raw_train_paths)}  "
      f"Val:{len(raw_val_paths)}  "
      f"Test:{len(raw_test_paths)}")

print("\n🔍 Running QA filter ...")
train_paths, train_labels, rej_tr = qa_filter(
    raw_train_paths, raw_train_labels)
val_paths,   val_labels,   rej_vl = qa_filter(
    raw_val_paths,   raw_val_labels)
test_paths,  test_labels,  rej_ts = qa_filter(
    raw_test_paths,  raw_test_labels)

print(f"   Train → kept {len(train_paths):>5}, rejected {rej_tr}")
print(f"   Val   → kept {len(val_paths):>5}, rejected {rej_vl}")
print(f"   Test  → kept {len(test_paths):>5}, rejected {rej_ts}")

In [ ]:
# ============================================================
#  CELL 4 — TRANSFORMS, DATASET & DATALOADERS
# ============================================================

# Training: augmentation to simulate scanner/patient variance
# Val/Test: clean resize + normalise only
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ChestXRayDataset(Dataset):
    """Loads X-Ray images from path lists and applies transforms."""
    def __init__(self, paths, labels, transform=None):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img   = Image.open(self.paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset = ChestXRayDataset(
    train_paths, train_labels, train_transform)
val_dataset   = ChestXRayDataset(
    val_paths,   val_labels,   eval_transform)
test_dataset  = ChestXRayDataset(
    test_paths,  test_labels,  eval_transform)

# Inverse-frequency class weights
counts        = Counter(train_labels)
n_total       = len(train_labels)
class_weights = torch.tensor(
    [n_total / (2 * counts[i]) for i in range(2)],
    dtype=torch.float32
).to(DEVICE)

print("⚖️  Class weights:")
for i, w in enumerate(class_weights):
    print(f"   {IDX_TO_CLASS[i]:<12}: {w.item():.4f}")

# WeightedRandomSampler for balanced training batches
sample_weights = [1.0 / counts[l] for l in train_labels]
sampler        = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float64),
    num_samples = len(train_labels),
    replacement = True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS)
val_loader   = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS)

print(f"\n✅ DataLoaders ready")
print(f"   Train batches : {len(train_loader)}")
print(f"   Val   batches : {len(val_loader)}")
print(f"   Test  batches : {len(test_loader)}")

In [ ]:
# ============================================================
#  CELL 5 — EDA DASHBOARD
# ============================================================

fig = plt.figure(figsize=(20, 12), facecolor="#0E1117")
fig.suptitle("EDA — Chest X-Ray Pneumonia Dataset",
             color="white", fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(2, 4, figure=fig,
                        hspace=0.45, wspace=0.38)

# Panel 1: Train class distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor("#1A1D23")
cnts = [train_labels.count(0), train_labels.count(1)]
bars = ax1.bar(CLASS_NAMES, cnts,
               color=[COLORS["NORMAL"], COLORS["PNEUMONIA"]],
               edgecolor="#222", width=0.5)
for bar, c in zip(bars, cnts):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 20, f"{c}",
             ha="center", color="white",
             fontsize=10, fontweight="bold")
ax1.set_title("Train Class Counts", color="white", fontsize=10)
ax1.tick_params(colors="#AAAAAA")
for sp in ["top","right"]:   ax1.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax1.spines[sp].set_color("#444")

# Panel 2: QA rejections
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor("#1A1D23")
ax2.bar(["Train","Val","Test"], [rej_tr, rej_vl, rej_ts],
        color="#E74C3C", edgecolor="#222", width=0.4)
for i, v in enumerate([rej_tr, rej_vl, rej_ts]):
    ax2.text(i, v + 0.3, str(v), ha="center",
             color="white", fontsize=10, fontweight="bold")
ax2.set_title("QA Rejected Images", color="white", fontsize=10)
ax2.tick_params(colors="#AAAAAA")
for sp in ["top","right"]:   ax2.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax2.spines[sp].set_color("#444")

# Panel 3: Laplacian score histogram
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor("#1A1D23")
lap_scores = {"NORMAL": [], "PNEUMONIA": []}
for path, label in zip(train_paths[:500], train_labels[:500]):
    score = laplacian_variance(path)
    lap_scores[IDX_TO_CLASS[label]].append(score)
for cls in CLASS_NAMES:
    ax3.hist(lap_scores[cls], bins=40, alpha=0.6,
             color=COLORS[cls], label=cls, density=True)
ax3.axvline(QA_BLUR_THRESHOLD, color="#FFD700",
            linewidth=1.5, linestyle="--",
            label=f"Threshold={QA_BLUR_THRESHOLD}")
ax3.set_title("Laplacian Score Distribution",
              color="white", fontsize=10)
ax3.set_xlabel("Sharpness", color="#AAAAAA", fontsize=8)
ax3.legend(facecolor="#1A1D23", edgecolor="#444",
           labelcolor="white", fontsize=7)
ax3.tick_params(colors="#AAAAAA")
for sp in ["top","right"]:   ax3.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax3.spines[sp].set_color("#444")

# Panel 4: Pixel intensity histogram
ax4 = fig.add_subplot(gs[0, 3])
ax4.set_facecolor("#1A1D23")
for cls, lbl in CLASS_TO_IDX.items():
    px_vals = []
    for p in [x for x, l in zip(train_paths, train_labels)
               if l == lbl][:150]:
        g = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if g is not None:
            px_vals.extend(
                cv2.resize(g,(64,64)).flatten().tolist())
    if px_vals:
        ax4.hist(px_vals, bins=64, alpha=0.5,
                 color=COLORS[cls], label=cls, density=True)
ax4.set_title("Pixel Intensity", color="white", fontsize=10)
ax4.set_xlabel("Value", color="#AAAAAA", fontsize=8)
ax4.legend(facecolor="#1A1D23", edgecolor="#444",
           labelcolor="white", fontsize=7)
ax4.tick_params(colors="#AAAAAA")
for sp in ["top","right"]:   ax4.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax4.spines[sp].set_color("#444")

# Panels 5-8: Sample X-Rays with CLAHE
sample_show = {
    "NORMAL":    [p for p,l in zip(train_paths,train_labels) if l==0][:2],
    "PNEUMONIA": [p for p,l in zip(train_paths,train_labels) if l==1][:2],
}
col_idx = 0
for cls, spaths in sample_show.items():
    for path in spaths:
        ax = fig.add_subplot(gs[1, col_idx])
        gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))
        clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        combined = np.hstack([gray, enhanced])
        ax.imshow(combined, cmap="bone")
        lap = laplacian_variance(path)
        ax.set_title(f"{cls}\nRaw | CLAHE  (Lap={lap:.0f})",
                     color=COLORS[cls], fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor(COLORS[cls])
            spine.set_linewidth(2)
        col_idx += 1

fig.patch.set_facecolor("#0E1117")
plt.savefig("eda_dashboard.png", dpi=150,
            bbox_inches="tight", facecolor="#0E1117")
plt.show()
print("🖼  Saved: eda_dashboard.png")

In [ ]:
# ============================================================
#  CELL 6 — BUILD DENSENET121 MODEL
# ============================================================

def build_model():
    """
    DenseNet121 pretrained on ImageNet.
    Strategy:
      · Freeze all layers     → keep generic features
      · Unfreeze denseblock4  → fine-tune on X-Ray data
      · New classifier head   → 2-class output
    """
    m = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT)

    # Freeze all layers
    for p in m.parameters():
        p.requires_grad = False

    # Unfreeze last dense block for fine-tuning
    for p in m.features.denseblock4.parameters():
        p.requires_grad = True

    # New classifier: 1024 → 256 → 2
    in_f = m.classifier.in_features
    m.classifier = nn.Sequential(
        nn.Linear(in_f, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, NUM_CLASSES)
    )
    return m.to(DEVICE)


model     = build_model()
optimizer = Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=1e-5
)
scheduler = ReduceLROnPlateau(
    optimizer, mode="min",
    factor=0.5, patience=3, verbose=False
)
criterion = nn.CrossEntropyLoss(
    weight=class_weights, label_smoothing=0.05)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad)

print("✅ DenseNet121 ready")
print(f"   Total params    : {total:,}")
print(f"   Trainable       : {trainable:,}")
print(f"   Frozen          : {total - trainable:,}")

In [ ]:
# ============================================================
#  CELL 7 — TRAINING LOOP WITH EARLY STOPPING
# ============================================================

def train_epoch(model, loader, optimizer, criterion):
    """One full pass over training data. Returns loss, accuracy."""
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, lbls in tqdm(loader, desc="  Train", leave=False):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        logits     = model(imgs)
        loss       = criterion(logits, lbls)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == lbls).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


def eval_epoch(model, loader, criterion):
    """One full pass over val/test data — no gradients."""
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="  Val", leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits     = model(imgs)
            loss       = criterion(logits, lbls)
            loss_sum  += loss.item() * imgs.size(0)
            correct   += (logits.argmax(1)==lbls).sum().item()
            total     += imgs.size(0)
    return loss_sum / total, correct / total


best_val_loss = float("inf")
no_improve    = 0
best_weights  = None
history       = {"tl":[], "ta":[], "vl":[], "va":[]}

print(f"🚀 Training  (max {NUM_EPOCHS} epochs, "
      f"early stop patience={PATIENCE})")
print("=" * 62)

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tl, ta = train_epoch(model, train_loader,
                          optimizer, criterion)
    vl, va = eval_epoch(model, val_loader, criterion)
    scheduler.step(vl)

    history["tl"].append(tl); history["ta"].append(ta)
    history["vl"].append(vl); history["va"].append(va)

    elapsed = time.time() - t0
    print(f"  Ep {epoch:>2}/{NUM_EPOCHS} | "
          f"TrLoss:{tl:.4f} TrAcc:{ta*100:.1f}% | "
          f"VlLoss:{vl:.4f} VlAcc:{va*100:.1f}% | "
          f"{elapsed:.0f}s")

    if vl < best_val_loss:
        best_val_loss = vl
        no_improve    = 0
        best_weights  = copy.deepcopy(model.state_dict())
        torch.save(best_weights, "medical_model.pth")
        print(f"    💾 Best saved  (val_loss={vl:.4f})")
    else:
        no_improve += 1
        print(f"    ⏳ No improve  ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"  ⛔ Early stop at epoch {epoch}")
            break

model.load_state_dict(
    torch.load("medical_model.pth", map_location=DEVICE))
model.eval()
print("\n✅ Best model reloaded → medical_model.pth")

In [ ]:
# ============================================================
#  CELL 8 — TRAINING CURVES
# ============================================================

ep_range = range(1, len(history["tl"]) + 1)
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(13, 4), facecolor="#0E1117")

for ax in [ax1, ax2]:
    ax.set_facecolor("#1A1D23")
    ax.tick_params(colors="#AAAAAA")
    for sp in ["top","right"]:   ax.spines[sp].set_visible(False)
    for sp in ["bottom","left"]: ax.spines[sp].set_color("#444")

ax1.plot(ep_range, history["tl"], "#4A90E2",
         linewidth=2, label="Train")
ax1.plot(ep_range, history["vl"], "#E74C3C",
         linewidth=2, label="Val")
ax1.set_title("Loss", color="white", fontsize=11)
ax1.set_xlabel("Epoch", color="#AAAAAA")
ax1.set_ylabel("Loss",  color="#AAAAAA")
ax1.legend(facecolor="#1A1D23", edgecolor="#444",
           labelcolor="white")

ax2.plot(ep_range, [a*100 for a in history["ta"]],
         "#2ECC71", linewidth=2, label="Train")
ax2.plot(ep_range, [a*100 for a in history["va"]],
         "#FFD700", linewidth=2, label="Val")
ax2.set_title("Accuracy (%)", color="white", fontsize=11)
ax2.set_xlabel("Epoch",       color="#AAAAAA")
ax2.set_ylabel("Accuracy %",  color="#AAAAAA")
ax2.legend(facecolor="#1A1D23", edgecolor="#444",
           labelcolor="white")

fig.suptitle("Training Curves", color="white",
             fontsize=13, fontweight="bold")
fig.patch.set_facecolor("#0E1117")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150,
            bbox_inches="tight", facecolor="#0E1117")
plt.show()
print("🖼  Saved: training_curves.png")

In [ ]:
# ============================================================
#  CELL 9 — TEST EVALUATION & METRICS
# ============================================================

def run_inference(model, loader):
    """Run the full test set through the model."""
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="Testing"):
            imgs   = imgs.to(DEVICE)
            logits = model(imgs)
            probs  = torch.softmax(logits, dim=1)
            preds  = logits.argmax(dim=1)
            y_true.extend(lbls.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs[:,1].cpu().numpy())
    return (np.array(y_true),
            np.array(y_pred),
            np.array(y_prob))


y_true, y_pred, y_prob = run_inference(model, test_loader)

cm           = confusion_matrix(y_true, y_pred)
TN,FP,FN,TP  = cm.ravel()
accuracy     = (TP+TN) / len(y_true)
sensitivity  = TP / (TP+FN)
specificity  = TN / (TN+FP)
f1           = f1_score(y_true, y_pred, pos_label=1)
roc_auc      = roc_auc_score(y_true, y_prob)

print(f"\n📊 Test Set Metrics")
print(f"{'='*45}")
print(f"  Accuracy    : {accuracy*100:.2f}%")
print(f"  Sensitivity : {sensitivity*100:.2f}%  ← catches PNEUMONIA")
print(f"  Specificity : {specificity*100:.2f}%  ← avoids false alarms")
print(f"  F1 Score    : {f1:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  TP={TP} TN={TN} FP={FP} FN={FN}")
print(f"{'='*45}")

metrics = {
    "accuracy":    float(accuracy),
    "sensitivity": float(sensitivity),
    "specificity": float(specificity),
    "f1":          float(f1),
    "roc_auc":     float(roc_auc),
    "cm":          cm.tolist()
}
with open("eval_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("💾 Saved: eval_metrics.json")

In [ ]:
# ============================================================
#  CELL 10 — CONFUSION MATRIX & ROC CURVE
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(11, 4), facecolor="#0E1117")

# Confusion Matrix
ax1.set_facecolor("#1A1D23")
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
cell_c = [["#1A4731","#4A1A1A"],["#4A1A1A","#1A4731"]]
for r in range(2):
    for c in range(2):
        ax1.add_patch(plt.Rectangle(
            (c,r), 1, 1, color=cell_c[r][c]))
        ax1.text(c+.5, r+.5,
                 f"{cm[r,c]}\n({cm_pct[r,c]:.1f}%)",
                 ha="center", va="center",
                 color="white", fontsize=13, fontweight="bold")
ax1.set_xlim(0,2); ax1.set_ylim(0,2)
ax1.set_xticks([.5,1.5]); ax1.set_yticks([.5,1.5])
ax1.set_xticklabels(CLASS_NAMES, color="white")
ax1.set_yticklabels(CLASS_NAMES, color="white")
ax1.set_xlabel("Predicted", color="#AAAAAA")
ax1.set_ylabel("True",      color="#AAAAAA")
ax1.set_title("Confusion Matrix", color="white", fontweight="bold")
for (r,c), lbl in [((0,0),"TN"),((0,1),"FP"),
                    ((1,0),"FN"),((1,1),"TP")]:
    ax1.text(c+.88, r+.12, lbl, color="#AAAAAA", fontsize=8)

# ROC Curve
ax2.set_facecolor("#1A1D23")
fpr, tpr, _ = roc_curve(y_true, y_prob)
ax2.plot(fpr, tpr, "#4A90E2", linewidth=2.5,
         label=f"AUC = {roc_auc:.4f}")
ax2.fill_between(fpr, tpr, alpha=0.15, color="#4A90E2")
ax2.plot([0,1],[0,1],"#666",linestyle="--",label="Random (0.5)")
ax2.set_xlabel("False Positive Rate", color="#AAAAAA")
ax2.set_ylabel("True Positive Rate",  color="#AAAAAA")
ax2.set_title("ROC Curve", color="white", fontweight="bold")
ax2.legend(facecolor="#1A1D23", edgecolor="#444",
           labelcolor="white")
ax2.tick_params(colors="#AAAAAA")
for sp in ["top","right"]:   ax2.spines[sp].set_visible(False)
for sp in ["bottom","left"]: ax2.spines[sp].set_color("#444")

fig.patch.set_facecolor("#0E1117")
plt.tight_layout()
plt.savefig("confusion_roc.png", dpi=150,
            bbox_inches="tight", facecolor="#0E1117")
plt.show()
print("🖼  Saved: confusion_roc.png")

In [ ]:
# ============================================================
#  CELL 11 — GRAD-CAM VISUALISATION
# ============================================================

class GradCAM:
    """
    Generates attention heatmaps using PyTorch hooks.
    Forward hook  → saves feature maps (activations)
    Backward hook → saves gradients
    Weighted sum → ReLU → resize → overlay on X-Ray
    """
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(
            lambda m,i,o:
                setattr(self, "activations", o.detach()))
        target_layer.register_full_backward_hook(
            lambda m,gi,go:
                setattr(self, "gradients", go[0].detach()))

    def generate(self, tensor, class_idx=None):
        self.model.eval()
        tensor = tensor.to(DEVICE)
        logits = self.model(tensor)
        probs  = torch.softmax(logits, dim=1)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        conf = probs[0, class_idx].item()
        self.model.zero_grad()
        logits[0, class_idx].backward()
        weights = self.gradients[0].mean(dim=(1, 2))
        cam = torch.zeros(
            self.activations.shape[2:], device=DEVICE)
        for i, w in enumerate(weights):
            cam += w * self.activations[0, i]
        cam = F.relu(cam)
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        cam = cv2.resize(cam.cpu().numpy(),
                         (IMG_SIZE, IMG_SIZE))
        return cam, class_idx, conf


gradcam = GradCAM(model, model.features.denseblock4)
print("✅ Grad-CAM attached to denseblock4")


def visualise_gradcam(test_paths, test_labels, gradcam, n=4):
    """Show Original | Heatmap | Overlay for n test images."""
    norm_p = [p for p,l in zip(test_paths,test_labels) if l==0]
    pneu_p = [p for p,l in zip(test_paths,test_labels) if l==1]
    paths  = (random.sample(norm_p, min(n//2, len(norm_p))) +
               random.sample(pneu_p, min(n//2, len(pneu_p))))
    lbls   = ([0]*min(n//2, len(norm_p)) +
               [1]*min(n//2, len(pneu_p)))

    fig, axes = plt.subplots(
        len(paths), 3,
        figsize=(12, len(paths)*3.8),
        facecolor="#0E1117")
    fig.suptitle("Grad-CAM — Lung Attention Heatmaps",
                 color="white", fontsize=13, fontweight="bold")

    for col, title in enumerate(["Original X-Ray",
                                   "Grad-CAM Heatmap",
                                   "Overlay"]):
        axes[0,col].set_title(title, color="#AAAAAA",
                               fontsize=10, pad=8)

    for row, (path, true_lbl) in enumerate(zip(paths, lbls)):
        tensor  = eval_transform(
            Image.open(path).convert("RGB")
        ).unsqueeze(0)
        heatmap, pred_idx, conf = gradcam.generate(tensor)

        orig    = cv2.imread(path)
        orig    = cv2.resize(orig, (IMG_SIZE, IMG_SIZE))
        orig_r  = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
        hmap_u8 = np.uint8(255 * heatmap)
        hmap_c  = cv2.applyColorMap(hmap_u8, cv2.COLORMAP_JET)
        hmap_c  = cv2.cvtColor(hmap_c, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(orig_r, 0.55, hmap_c, 0.45, 0)

        true_cls = IDX_TO_CLASS[true_lbl]
        pred_cls = IDX_TO_CLASS[pred_idx]
        ok       = true_lbl == pred_idx
        color    = "#2ECC71" if ok else "#E74C3C"

        axes[row,0].set_ylabel(
            f"{'✅' if ok else '❌'} True:{true_cls}\n"
            f"   Pred:{pred_cls} ({conf*100:.0f}%)",
            color=color, fontsize=8,
            rotation=0, labelpad=90, va="center")

        for col, img in enumerate([orig_r, hmap_c, overlay]):
            axes[row,col].imshow(img)
            axes[row,col].set_xticks([])
            axes[row,col].set_yticks([])
            for sp in axes[row,col].spines.values():
                sp.set_edgecolor(color)
                sp.set_linewidth(1.5)

    fig.patch.set_facecolor("#0E1117")
    plt.tight_layout()
    plt.savefig("gradcam_results.png", dpi=150,
                bbox_inches="tight", facecolor="#0E1117")
    plt.show()
    print("🖼  Saved: gradcam_results.png")


visualise_gradcam(test_paths, test_labels, gradcam, n=4)

In [ ]:
# ============================================================
#  CELL 12 — FINAL SUMMARY & FILE CHECKLIST
# ============================================================

files_produced = [
    "medical_model.pth",
    "eval_metrics.json",
    "eda_dashboard.png",
    "training_curves.png",
    "confusion_roc.png",
    "gradcam_results.png",
]

print("=" * 52)
print("  ✅  TRAINING PIPELINE COMPLETE")
print("=" * 52)
print(f"  Accuracy    : {accuracy*100:.2f}%")
print(f"  Sensitivity : {sensitivity*100:.2f}%")
print(f"  Specificity : {specificity*100:.2f}%")
print(f"  F1 Score    : {f1:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print("-" * 52)
print("  Files saved:")
for fname in files_produced:
    exists = Path(fname).exists()
    status = "✅" if exists else "❌"
    print(f"    {status}  {fname}")
print("=" * 52)
print("")
print("  👉  Copy medical_model.pth and eval_metrics.json")
print("      into the same folder as the Deployment Notebook.")